# 第2回 発展: 用語集で LoRA 学習し、学習前後を比較する

講座の用語集（`glossary_sft.jsonl`、58 語 × 質問 2 通り = 116 行）を FAQ の代わりに使い、`lora_format_tuning.ipynb` と同じ手順で LoRA 学習と評価を行います。

望ましい出力は FAQ 版と同じ `{"answer": <文字列>, "needs_review": <真偽値>}` の JSON 1 行です。評価も同じく **形式遵守率**（JSON の形式を守れたか）と **内容正解率**（`needs_review` と答えの語句が正解と合うか）に分けます。

**FAQ 版との違い**

- プロンプトに FAQ（用語集）を入れません。用語集は 58 語あり、毎回渡すには長すぎるためです。用語の説明は **学習でモデルに覚えさせる** ことになります（第3回の「用語集を検索するエージェント」との比較点です）。
- 用語集の 116 行は、答えを `{"answer": <用語集の説明>, "needs_review": false}` に変換して学習します。用語集にない質問の例（12 件）を `needs_review: true` として足します。
- 評価の質問は 3 種類です。

| 種類 | 内容 | 正解 |
| --- | --- | --- |
| A | 用語集にある・学習で見た質問そのもの | `needs_review: false`、用語集の説明の語句を含む |
| B | 用語集にある用語を、学習で見ていない言い回しで聞く | `needs_review: false`、用語集の説明の語句を含む |
| C | 用語集にない質問（学習用の「用語集にない質問」とも別） | `needs_review: true`、answer に「確認」を含む |

**使うもの**: `Qwen/Qwen3.5-0.8B`、`glossary_sft.jsonl`（このノートブックと同じフォルダ。Colab では左のファイル一覧からアップロード）。Mac（MPS）、Colab（T4）のどちらでも動きます。

**読み方**: セルは上から順に実行してください。途中を飛ばすと、後のセルで変数が見つからないエラーになります。

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムや Mac には nvidia-smi が無い
    gpu_info = ""
import torch
if not gpu_info and torch.backends.mps.is_available():
    gpu_info = "Apple Silicon の GPU（MPS）を使います。"
print(gpu_info or "GPU が見つかりません。Colab ならランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

Apple Silicon の GPU（MPS）を使います。


In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/rakkotan/works/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.14.1'}


## 1. 設定

**やること**: モデル名、エポック数などを辞書 `CONFIG` にまとめます。セクション 7 の `result.json` にそのまま記録されます。

- `num_train_epochs: 5`: 学習データ 128 件（用語集 116 ＋ 用語集にない質問 12）を 5 周します。1 エポックは 16 ステップ（バッチ 4 × 勾配累積 2 = 8 件）で、合計 80 ステップです。
- `max_new_tokens: 256`: 用語集の説明は最長 212 文字あり、JSON にするとさらに長くなるので、FAQ 版（96）より長くしています。
- `glossary_path`: 用語集のファイル。Colab ではアップロード先の `/content/glossary_sft.jsonl` を見ます。

**見るところ**: 設定の JSON が表示されます。

In [3]:
from pathlib import Path
import os, sys, json, time

IN_COLAB = "google.colab" in sys.modules   # Colab 以外（Mac など）ではローカルのフォルダを使う

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 5,   # 学習データ（128 件）を何周学習するか。1 周 16 ステップなので合計 80 ステップ
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "glossary_path": "/content/glossary_sft.jsonl" if IN_COLAB else "glossary_sft.jsonl",
    "work_dir": "/content/glossary_lab" if IN_COLAB else "glossary_lab",   # 結果ファイルの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 5,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "glossary_path": "glossary_sft.jsonl",
  "work_dir": "glossary_lab"
}


## 2. 用語集の読み込みと教師データ・評価データの作成

**やること**: 用語集を JSON 形式の教師データに変換し、評価用の質問（test）と正解（gold）を作ります。

- 用語集にある質問: answer は用語集の説明そのまま、`needs_review: false`。
- 用語集にない質問: answer は「この用語集には記載がありません。確認が必要です。」、`needs_review: true`。学習用に 12 件（`UNGROUNDED_TRAIN`）用意します。
- test は 24 問です。A（学習した質問）8、B（言い換え）8、C（用語集にない）8。C は学習用の 12 件とは別の質問です。
- 正解（`needs_review` の値と、answer に含まれるべき語句）は `gold` に分けて持ちます。A / B の語句が用語集の説明に実在することはセルの中で確かめます。

学習データは用語集にある質問 116 件に対し、用語集にない質問が 12 件と偏っています。FAQ 版（根拠あり 10、根拠なし 4）より「ない」の割合が小さいことが、C の結果にどう出るかも見どころです。

**見るところ**: `train=128 test=24 (A=8 B=8 C=8) gold=24` と、学習例の 1 件が表示されます。質問が train と test（B / C）で重複していると、ここでエラーになります。

In [4]:
path = Path(CONFIG["glossary_path"])
assert path.exists(), f"{path} が見つかりません。ノートブックと同じフォルダ（Colab ならアップロード先）に置いてください"
glossary = [json.loads(l)["messages"] for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
REF = {m[0]["content"]: m[1]["content"] for m in glossary}   # 質問 → 用語集の説明

UNGROUNDED_ANSWER = "この用語集には記載がありません。確認が必要です。"
# 学習用の「用語集にない質問」（test の C とは別）
UNGROUNDED_TRAIN = [
    "Kubernetes とは何ですか？", "強化学習とは何ですか？", "CNN について簡単に説明してください。", "勾配降下法とは何ですか？",
    "SQL について簡単に説明してください。", "BERT とは何ですか？", "GAN について簡単に説明してください。", "正規表現とは何ですか？",
    "ドロップアウトとは何ですか？", "活性化関数について簡単に説明してください。", "この講座の受講料はいくらですか？", "第10回の講義テーマは何ですか？",
]

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "target": target, "needs_review": needs_review}

train = [make_example(q, a, False) for q, a in REF.items()] + [make_example(q, UNGROUNDED_ANSWER, True) for q in UNGROUNDED_TRAIN]

# A: 用語集にある・学習した質問そのもの
A = [("「トークン」とは何ですか？", ["最小単位"]), ("「temperature」とは何ですか？", ["ランダム"]),
     ("「ハルシネーション」とは何ですか？", ["もっともらしく"]), ("「LoRA」とは何ですか？", ["低ランク"]),
     ("「過学習」とは何ですか？", ["未知"]), ("「量子化」とは何ですか？", ["ビット"]),
     ("「エポック」とは何ですか？", ["1周"]), ("「ガードレール」とは何ですか？", ["制約"])]
# B: 用語集にある用語を、学習で見ていない言い回しで聞く（第 3 要素は同じ用語の学習済みの質問）
B = [("APIキーは何のための文字列ですか？", ["秘密"], "「APIキー」とは何ですか？"),
     ("学習率とは何を決める値ですか？", ["ハイパーパラメータ"], "「学習率」とは何ですか？"),
     ("MCP って何をするためのものですか？", ["標準プロトコル"], "「MCP（Model Context Protocol）」とは何ですか？"),
     ("エージェントループの仕組みを教えてください。", ["ツール"], "「エージェントループ」とは何ですか？"),
     ("LLM の出力を JSON などの決まった形で受け取る仕組みは何ですか？", ["JSON"], "「構造化出力」とは何ですか？"),
     ("人が途中で承認を挟む設計は何と呼びますか？", ["人の確認"], "「Human-in-the-loop」とは何ですか？"),
     ("Git のブランチは何のために使いますか？", ["main"], "「ブランチ」とは何ですか？"),
     ("LLM が一度に扱える長さの上限は何と呼びますか？", ["上限"], "「コンテキストウィンドウ」とは何ですか？")]
# C: 用語集にない質問（一般的な用語 6、講座固有だが載っていない事柄 2）
C = ["RAG とは何ですか？", "Transformer とは何ですか？", "RLHF とは何ですか？", "プロンプトインジェクションとは何ですか？",
     "ベクトルデータベースとは何ですか？", "Docker とは何ですか？", "この講座の最終課題の締切はいつですか？", "担当教員の名前は何ですか？"]

test, gold = [], {}
for q, words in A:
    assert q in REF and all(w in REF[q] for w in words), q
    test.append({**make_example(q, REF[q], False), "category": "A"}); gold[q] = {"needs_review": False, "must_include": words}
for q, words, ref in B:
    assert all(w in REF[ref] for w in words), q
    test.append({**make_example(q, REF[ref], False), "category": "B"}); gold[q] = {"needs_review": False, "must_include": words}
for q in C:
    test.append({**make_example(q, UNGROUNDED_ANSWER, True), "category": "C"}); gold[q] = {"needs_review": True, "must_include": ["確認"]}

train_questions = {r["question"] for r in train}
assert not [e["question"] for e in test if e["category"] != "A" and e["question"] in train_questions], "B / C の質問が train と重複しています"
counts = {k: sum(e["category"] == k for e in test) for k in "ABC"}
print(f"train={len(train)} test={len(test)} (A={counts['A']} B={counts['B']} C={counts['C']}) gold={len(gold)}")
print("学習例の 1 件 | 質問:", train[0]["question"], "| 望ましい出力:", train[0]["target"][:80], "...")
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

train=128 test=24 (A=8 B=8 C=8) gold=24
学習例の 1 件 | 質問: 「AIエンジニアリング」とは何ですか？ | 望ましい出力: {"answer": "学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを ...


## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 24 問に答えさせて採点します。採点のしかたは FAQ 版と同じです。

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句が `answer` に含まれること。

FAQ 版と違ってプロンプトに用語集を入れていないので、学習前のモデルは用語集の説明を知りません。A / B の内容は、モデルが元々持っている知識で偶然合うかどうかです。

**見るところ**

- 1 つ目のセル: モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate`、種類ごとの内容正解率 `content_A` など）と、24 問の出力の表。

**出ても問題ない警告**: `You are sending unauthenticated requests to the HF Hub`、`... is falling back to its reference PyTorch implementation ...`（FAQ 版の説明を参照）。

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。用語集はプロンプトに入れない（知識は学習で入れる）
SYSTEM = "あなたは AI エンジニアリング講座の用語集の回答係です。用語集の説明だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。用語集に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": example["question"]}]
def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Loading weights: 100%|██████████| 320/320 [00:01<00:00, 210.16it/s]


loaded: Qwen/Qwen3.5-0.8B | device: mps | 重みのメモリ(GB): 3.01


In [6]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の 24 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"category": ex["category"], "question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               **{f"content_{k}": sum(r["content_ok"] for r in rows if r["category"] == k) / counts[k] for k in "ABC"},
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は用語集と合っているかを読む
display(pd.DataFrame([{"種類": r["category"], "質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 24, "format_rate": 0.125, "content_rate": 0.0, "both_rate": 0.0, "content_A": 0.0, "content_B": 0.0, "content_C": 0.0, "seconds": 39.4}


,種類,質問,正解の needs_review,学習前の出力,形式,形式の失敗理由,内容
0,A,「トークン」とは何ですか？,False,"{""answer"": ""トークンとは、データやメッセージを解析する際に使用される単位です。例えば、JSON 解析では「{\""key\"": \""value\""}」というオブジェクトを「トークン」として扱うことが一般的です。""}",×,missing_key,×
1,A,「temperature」とは何ですか？,False,"{""answer"": ""温度は、物質の分子運動の速度や熱エネルギーを測る物理量です。""}",×,missing_key,×
2,A,「ハルシネーション」とは何ですか？,False,"{""answer"": ""振動が周期的に増減し、振幅が最大値から最小値へ変化する現象""}",×,missing_key,×
3,A,「LoRA」とは何ですか？,False,"{""answer"": ""低秩自编码器（Low-Rank Adaptation）の略称で、大規模モデルを効率的に学習するための技術です。""",×,json_error: JSONDecodeError,×
4,A,「過学習」とは何ですか？,False,"{""answer"": ""過学習とは、学習アルゴリズムが学習対象のデータに過度に習得し、テストデータや実装時に誤り（過学習）を起こす現象です。""}",×,missing_key,×
5,A,「量子化」とは何ですか？,False,"{""answer"": ""量子化とは、物理的な量（エネルギー、質量、速度など）を量子力学の枠組みで定義し、その値が離散的（分岐点）であることを指します。例えば、電子のエネルギーは量子力学で分岐点として定義され、連続的に変化しないという性質を持ちます。""}",×,missing_key,×
6,A,「エポック」とは何ですか？,False,"{""answer"": ""エポックは、プロジェクトの進捗状況や目標達成状況を示す重要な基準点です。プロジェクトが特定の目標を達成した瞬間に定義されます。""}",×,missing_key,×
7,A,「ガードレール」とは何ですか？,False,"{""answer"": ""ガードレールは、歩行者や車両が歩道や道路を安全に通行するための金属製のフェンスやブロックの構造物です。""}",×,missing_key,×
8,B,APIキーは何のための文字列ですか？,False,"{""answer"": ""認証とセキュリティのための一意のテキスト"", ""needs_review"": false}",○,,×
9,B,学習率とは何を決める値ですか？,False,"{""answer"": ""学習率（learning rate）は、モデルの学習速度を制御するパラメータで、各ステップでモデルが誤差を最小化するために必要なステップ数（ステップ数）を決定します。学習率の値は、モデルの複雑さやデータ量によって異なりますが、学習率が小さすぎると学習が遅くなり、学習率が大きすぎると過学習（過学習）や欠陥学習（欠陥学習）が発生する可能性があります。一般的に、学習率は 10^-4 から 10^-6 程度の値で調整されます。""}",×,missing_key,×


## 4. LoRA 学習

**やること**: train の 128 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 5 エポック（80 ステップ）学習します。学習した adapter は `adapter/` に保存します。LoRA の設定は FAQ 版と同じです（LoRA と QLoRA の違いは FAQ 版のセクション 4 を参照）。

- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の `model` で生成しても adapter 込みの出力になります。

**見るところ**

- `trainable params: ... trainable%: ...`: 学習するのは全体の数 % だけです。
- `Training Loss`（10 ステップごと）: 下がっていくこと。
- `学習時間: ... | ステップ数: 80 | エポック: 5.0`。Mac（MPS）では T4 より時間がかかることがあります。

**うまくいかないとき**: 途中で止まったら、`ckpt/` に 10 ステップごとのチェックポイントが残っています。セクション 0〜3 を実行し直し、下の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えると、続きから学習します。

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開はセクション 4 の説明を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

Truncating train dataset: 100%|██████████| 128/128 [00:00<00:00, 28540.26 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 128/128 [00:00<00:00, 81418.09 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
/Users/rakkotan/works/myenv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,2.360056
20,1.033048
30,0.786656
40,0.539728
50,0.395489
60,0.258070
70,0.218642
80,0.184696


/Users/rakkotan/works/myenv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/rakkotan/works/myenv/lib/python3.14/site-packages/peft/utils/other.py:1508: UserWarning: Unable to fetch remote file due to the following error [Errno 8] nodename nor servname provided, or not known - silently ignoring the lookup for the file config.json in Qwen/Qwen3.5-0.8B.
  warnings.warn(
/Users/rakkotan/works/myenv/lib/python3.14/site-packages/peft/utils/save_and_load.py:313: UserWarning: Could not find a config file in Qwen/Qwen3.5-0.8B - will assume that the vocabulary was not modified.
  warnings.warn(
/Users/rakkotan/works/myenv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/User

学習時間: 573 秒 | ステップ数: 80 | エポック: 5.0


## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 24 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。種類ごとの `content_A` / `content_B` / `content_C` も見ます。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - A が ○ でも B が ×: 言い換えに対応できず、学習した文を暗記しているだけかもしれません。
  - C（用語集にない）に `needs_review: false` で、用語集の別の用語の説明や、もっともらしい説明を当てて断定した。
  - A / B（用語集にある）に「記載がありません」と確認要求で逃げた。

seed を固定しても、実行環境が違えば結果は完全には再現しません。割合が似ていても、どの質問が残るかは実行ごとに変わることがあります。

In [8]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "content_A", "content_B", "content_C", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"種類": b["category"], "質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 24, "format_rate": 1.0, "content_rate": 0.5416666666666666, "both_rate": 0.5416666666666666, "content_A": 0.625, "content_B": 0.5, "content_C": 0.5, "seconds": 45.6}


,n,format_rate,content_rate,both_rate,content_A,content_B,content_C,seconds
label,,,,,,,,
before,24,0.125,0.000000,0.000000,0.000,0.0,0.0,39.4
after,24,1.000,0.541667,0.541667,0.625,0.5,0.5,45.6


退行: []


,種類,質問,正解の needs_review,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,A,「トークン」とは何ですか？,False,"{""answer"": ""トークンとは、データやメッセージを解析する際に使用される単位です。例えば、JSON 解析では「{\""key\"": \""value\""}」というオブジェクトを「トークン」として扱うことが一般的です。""}",×/×,"{""answer"": ""LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、検出ツール（フィードバック）の構成が決まるのがよくわかる。日本語は英語より1文字あたりのトークン数が多くなりやすい。"", ""needs_review"": false}",○/○
1,A,「temperature」とは何ですか？,False,"{""answer"": ""温度は、物質の分子運動の速度や熱エネルギーを測る物理量です。""}",×/×,"{""answer"": ""学習データに与えるランダムさのレベル。低いほど結果が安定しやすく、高いほど多様になる。同じ入力でも出力が変わる原因の一つ。"", ""needs_review"": false}",○/○
2,A,「ハルシネーション」とは何ですか？,False,"{""answer"": ""振動が周期的に増減し、振幅が最大値から最小値へ変化する現象""}",×/×,"{""answer"": ""LLMが事実と異なる内容や存在しない情報を、理由にもとから正しい情報に偽って返す現象。モデルが知らないこと（最新情報、存在しない組織・機関など）を、説明文に『知っている（検索履歴やツールの利用権限など）こと』と書き、表示したり、投稿したりする。本講座では、ツールを呼ばずに文章で答えていくため、ツールが知らずに返ってくるハルシネーションが多い。"", ""needs_review"": false}",○/×
3,A,「LoRA」とは何ですか？,False,"{""answer"": ""低秩自编码器（Low-Rank Adaptation）の略称で、大規模モデルを効率的に学習するための技術です。""",×/×,"{""answer"": ""この用語集には記載がありません。確認が必要です。"", ""needs_review"": true}",○/×
4,A,「過学習」とは何ですか？,False,"{""answer"": ""過学習とは、学習アルゴリズムが学習対象のデータに過度に習得し、テストデータや実装時に誤り（過学習）を起こす現象です。""}",×/×,"{""answer"": ""学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。"", ""needs_review"": false}",○/○
5,A,「量子化」とは何ですか？,False,"{""answer"": ""量子化とは、物理的な量（エネルギー、質量、速度など）を量子力学の枠組みで定義し、その値が離散的（分岐点）であることを指します。例えば、電子のエネルギーは量子力学で分岐点として定義され、連続的に変化しないという性質を持ちます。""}",×/×,"{""answer"": ""モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。"", ""needs_review"": false}",○/○
6,A,「エポック」とは何ですか？,False,"{""answer"": ""エポックは、プロジェクトの進捗状況や目標達成状況を示す重要な基準点です。プロジェクトが特定の目標を達成した瞬間に定義されます。""}",×/×,"{""answer"": ""学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。"", ""needs_review"": false}",○/○
7,A,「ガードレール」とは何ですか？,False,"{""answer"": ""ガードレールは、歩行者や車両が歩道や道路を安全に通行するための金属製のフェンスやブロックの構造物です。""}",×/×,"{""answer"": ""モデルの出力や行動を、特定の内容や形式、またはルールで検出してそれを拒絶する仕組み。Claude Codeの auto-code には Python でのループでコードを数行で書き、その結果をツールで受け取るように設定したのが標準のガードレール。"", ""needs_review"": false}",○/×
8,B,APIキーは何のための文字列ですか？,False,"{""answer"": ""認証とセキュリティのための一意のテキスト"", ""needs_review"": false}",○/×,"{""answer"": ""APIを呼び出す人の識別用秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、プロンプトに内包する。"", ""needs_review"": false}",○/○
9,B,学習率とは何を決める値ですか？,False,"{""answer"": ""学習率（learning rate）は、モデルの学習速度を制御するパラメータで、各ステップでモデルが誤差を最小化するために必要なステップ数（ステップ数）を決定します。学習率の値は、モデルの複雑さやデータ量によって異なりますが、学習率が小さすぎると学習が遅くなり、学習率が大きすぎると過学習（過学習）や欠陥学習（欠陥学習）が発生する可能性があります。一般的に、学習率は 10^-4 から 10^-6 程度の値で調整されます。""}",×/×,"{""answer"": ""1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。"", ""needs_review"": false}",○/○


## 6. 用語集にない質問（C）の応答を分類する

**やること**: C の 8 問の応答を、FAQ 版のセクション 6 と同じ 3 つに分類して、学習前後を並べます。

| 分類 | 意味 |
| --- | --- |
| `format_error` | JSON の形式を守れていない |
| `needs_review` | 「記載がありません。確認が必要です」と返した（望ましい応答） |
| `asserted` | `needs_review: false` で言い切った。用語集に無い質問なので、根拠のない断定です |

**見るところ**: 断定した場合、用語集の別の用語の説明を流用しているか、もっともらしい説明を作っているかを、出力の原文で読んでください。「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。

In [9]:
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

knowledge_rows = [{"質問": b["question"], "学習前の分類": classify(b["raw"]), "学習前の出力": b["raw"],
                   "学習後の分類": classify(a["raw"]), "学習後の出力": a["raw"]}
                  for b, a in zip(before_rows, after_rows) if b["category"] == "C"]
display(pd.DataFrame(knowledge_rows))
display(pd.DataFrame(knowledge_rows)[["学習前の分類", "学習後の分類"]].apply(lambda s: s.value_counts()).fillna(0).astype(int))

,質問,学習前の分類,学習前の出力,学習後の分類,学習後の出力
0,RAG とは何ですか？,format_error,"{""answer"": ""RAG (Retrieval-Augmented Generation) は、大規模なテキストベースの知識ベース（RAG）を用いて、ユーザーが質問する文脈に基づいて、関連する知識を抽出し、その知識に基づいて生成する技術です。""}",asserted,"{""answer"": ""モデルの質問応答先から、後続のツールの引用リンクをモデルに渡して、モデルがそのリンクから必要な情報を検索し、結果を返す、という「入力－検索－出力」のループをモデルに渡す技術。検索の結果が固定されたデータベースではなく、ツール呼び出しのリンクから実際のデータを取得できるため、利用者の行動をみる（どのツールを使いつか、どこに載る）と、結果が変わる。"", ""needs_review"": false}"
1,Transformer とは何ですか？,format_error,"{""answer"": ""Transformer は、機械学習の分野で広く使われる、高度なアルゴリズムの一つです。特に、大規模なデータ処理や、複雑な最適化問題において、非常に効果的な手法として知られています。""}",needs_review,"{""answer"": ""この用語集には記載がありません。確認が必要です。"", ""needs_review"": true}"
2,RLHF とは何ですか？,format_error,"{""answer"": ""RLHF は Reinforcement Learning from Human Feedback (人間からの報酬学習) の略です。AI モデルを学習する際に、人間がモデルの出力に対して評価し、その評価に基づいて報酬（スコア）を返すことで、モデルが人間が好む表現を学習するプロセスを指します。""}",asserted,"{""answer"": ""エージェントの出力を人の意見や評価で改善するループの前半部分。PPOやDPOなどのアルゴリズムで、エージェントの振る舞い（プロンプト）や出力を利用した人への説明文を、エージェントに渡して評価した人の感想を、エージェントの次の行動を誘導する。本講座の第3回では、このループを回すたびにモデルをより人への配慮のある振る舞いを身につけさせる（PPO-RLHF）。"", ""needs_review"": false}"
3,プロンプトインジェクションとは何ですか？,format_error,"{""answer"": ""プロンプトインジェクションとは、攻撃者が AI エンジニアリング講座の用語集の説明を根拠に、特定の用語集の定義や構造を悪用して、攻撃者の目的（例：情報漏洩、システム不正利用）を実現する手法を指します。""}",asserted,"{""answer"": ""モデルが指示を出す前に、モデルの性格や知識をツール名の後に書き、呼び分时で受け入れる攻撃。本講座では「ツール名の性格を、呼び分时で切り替えて、次のツールを間違えて使う」のが典型例。本講座では習わない。"", ""needs_review"": false}"
4,ベクトルデータベースとは何ですか？,format_error,"{""answer"": ""ベクトルデータベースは、データがベクトル（数値列）の形式で保存・操作されるデータベースです。ベクトルは、長さの異なる数値列であり、ベクトルデータベースは、これらのベクトルを効率的に検索・操作・管理するシステムです。""}",needs_review,"{""answer"": ""この用語集には記載がありません。確認が必要です。"", ""needs_review"": true}"
5,Docker とは何ですか？,format_error,"{""answer"": ""Docker は、容器化技術の一種で、アプリケーションを軽量なパッケージとして、OS の任意の環境で起動・停止・リロードできる環境を構築する技術です。""}",asserted,"{""answer"": ""プログラムを起動し、その起動後の状態を共有できる容器。内部の環境と構成ファイルで分割し、外側から切り離すためのプラットフォーム。"", ""needs_review"": false}"
6,この講座の最終課題の締切はいつですか？,asserted,"{""answer"": ""2024 年 10 月 15 日"", ""needs_review"": false}",needs_review,"{""answer"": ""この用語集には記載がありません。確認が必要です。"", ""needs_review"": true}"
7,担当教員の名前は何ですか？,format_error,"{""answer"": ""担当教員の名前は「未指定」です。""}",needs_review,"{""answer"": ""この用語集には記載がありません。確認が必要です。"", ""needs_review"": true}"


,学習前の分類,学習後の分類
asserted,1,4
format_error,7,0
needs_review,0,4


## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`（`glossary_lab/` にあります。Colab なら左のファイル一覧で右クリック → ダウンロード）
- セクション 5 の比較表と、A / B / C の違いについての説明
- セクション 6 の結果（確認要求に変わった応答、断定が残った応答）
- 第3回の「用語集を検索するエージェント」と比べて、用語集の質問にはどちらが向いているかの考察

In [10]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "glossary_path")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "data": {"train": len(train), "train_ungrounded": len(UNGROUNDED_TRAIN), "test": counts},
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.14.1"
  },
  "method": "LoRA（量子化なし）",
  "data": {
    "train": 128,
    "train_ungrounded": 12,
    "test": {
      "A": 8,
      "B": 8,
      "C": 8
    }
  },
  "before": {
    "label": "before",
    "n": 24,
    "format_rate": 0.125,
    "content_rate": 0.0,
    "both_rate": 0.0,
    "content_A": 0.0,
    "content_B": 0.0,
    "content_C": 0.0,
    "seconds": 39.4
  },
  "after": {
    "label": "after",
    "n": 24,
    "format_rate": 1.0,
    "content_rate": 0.5416666666666666,
    "both_rate": 0.5416666666666666,
    "content_A": 0.625,
    "co